In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urlparse
from pathlib import Path

urls = [
    "https://www.nccih.nih.gov/health/diabetes-and-dietary-supplements-what-you-need-to-know"
]

output_dir = Path("nutrition_scraped_data")
output_dir.mkdir(exist_ok=True)

nutrition_keywords = [
    "nutrition", "nutrient", "protein", "carbohydrate", "carb",
    "fat", "fiber", "fibre", "calcium", "iron", "zinc",
    "magnesium", "potassium", "sodium", "vitamin", "vitamins", "mineral",
    "rda", "rdi", "ear", "ai", "dietary", "diabetes",
    "glycemic", "energy", "calorie", "intake", "requirement"
]

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; NutritionResearchScraper/1.0)"
}

paragraph_records = []
list_records = []
table_records = []
errors = []

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def is_nutrition_related(text):
    text = text.lower()
    return any(
        re.search(r"\b" + re.escape(word) + r"\b", text)
        for word in nutrition_keywords
    )

for url in urls:
    print("Scraping:", url)

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "").lower()
        if "html" not in content_type:
            print("Skipped: URL did not return HTML.")
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript", "svg"]):
            tag.decompose()

        page_title = soup.title.get_text(" ", strip=True) if soup.title else ""

        for paragraph in soup.find_all("p"):
            text = clean_text(paragraph.get_text(" ", strip=True))
            if text and is_nutrition_related(text):
                paragraph_records.append({
                    "source_url": url,
                    "page_title": page_title,
                    "content_type": "paragraph",
                    "content": text
                })

        for list_tag in soup.find_all(["ol", "ul"]):
            items = [
                clean_text(item.get_text(" ", strip=True))
                for item in list_tag.find_all("li", recursive=False)
            ]
            items = [item for item in items if item]

            combined_text = " | ".join(items)
            if combined_text and is_nutrition_related(combined_text):
                list_records.append({
                    "source_url": url,
                    "page_title": page_title,
                    "list_type": "ordered" if list_tag.name == "ol" else "unordered",
                    "items": " | ".join(items)
                })

        for table_number, table in enumerate(soup.find_all("table"), start=1):
            try:
                dataframes = pd.read_html(str(table))
                if not dataframes:
                    continue

                df = dataframes[0]
                table_text = clean_text(df.astype(str).to_string(index=False))

                if is_nutrition_related(table_text):
                    table_records.append({
                        "source_url": url,
                        "page_title": page_title,
                        "table_number": table_number,
                        "table_html": str(table),
                        "table_text": table_text
                    })
            except Exception as table_error:
                print("Could not parse a table:", table_error)

    except requests.RequestException as error:
        errors.append({"source_url": url, "error": str(error)})
        print("Failed:", error)

pd.DataFrame(paragraph_records).to_csv(
    output_dir / "paragraphs.csv", index=False, encoding="utf-8-sig"
)
pd.DataFrame(list_records).to_csv(
    output_dir / "lists.csv", index=False, encoding="utf-8-sig"
)
pd.DataFrame(table_records).to_csv(
    output_dir / "tables.csv", index=False, encoding="utf-8-sig"
)
pd.DataFrame(errors).to_csv(
    output_dir / "errors.csv", index=False, encoding="utf-8-sig"
)

print("Scraping completed.")
print("Paragraphs:", len(paragraph_records))
print("Lists:", len(list_records))
print("Tables:", len(table_records))
print("Errors:", len(errors))
print("Output folder:", output_dir.resolve())

In [ ]:
paragraph_df = pd.DataFrame(paragraph_records)
list_df = pd.DataFrame(list_records)
table_df = pd.DataFrame(table_records)

combined_records = []

for _, row in paragraph_df.iterrows():
    combined_records.append({
        "source_url": row["source_url"],
        "page_title": row["page_title"],
        "content_type": "paragraph",
        "content": row["content"]
    })

for _, row in list_df.iterrows():
    combined_records.append({
        "source_url": row["source_url"],
        "page_title": row["page_title"],
        "content_type": row["list_type"] + " list",
        "content": row["items"]
    })

for _, row in table_df.iterrows():
    combined_records.append({
        "source_url": row["source_url"],
        "page_title": row["page_title"],
        "content_type": "table",
        "content": row["table_text"]
    })

pd.DataFrame(combined_records).to_csv(
    output_dir / "all_nutrition_content.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Combined file saved.")